In [ ]:
import scanpy as sc
from pathlib import Path
processed = Path.cwd().parent / "data" / "processed"

In [ ]:
adata = sc.read_h5ad(processed / "00_combined_processed.h5ad")

In [ ]:
lin_adatas = [
    "01_immune.h5ad",
    "02_endothelial.h5ad",
    "03_epithelial.h5ad",
    "04_mesenchymal.h5ad",
    "05_nk_ilc.h5ad"
]

adata.obs["celltype"] = "unannotated"
adata.obs["celltype"] = adata.obs["celltype"].astype(object)

In [ ]:
for l in lin_adatas:
    lin_adata = sc.read_h5ad(processed / l)
    assert lin_adata.obs_names.isin(adata.obs_names).all(), f"{l}: barcodes not in adata"
    adata.obs.loc[lin_adata.obs_names, "celltype"] = lin_adata.obs["celltype"].astype(object).values

adata.obs["celltype"] = adata.obs["celltype"].astype("category")

In [ ]:
n_unann = (adata.obs["celltype"] == "unannotated").sum()
adata = adata[adata.obs["celltype"] != "unannotated"].copy()
print(f"Number of unannotated cells: {n_unann}")
print(adata.obs["celltype"].value_counts())

adata.write_h5ad(processed / "06_combined_annotated.h5ad")